# Phase 5 — DLS Standard Edition Baseline and Validation
This notebook validates our coded implementation of the DLS Standard Edition resource table against matches in the dataset that had real-world DLS targets applied.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join('..')))
from src.common.config import INTERIM_DATA_DIR, FIGURES_DIR, TABLES_DIR
from src.common.dls_lookup import get_resource_pct, calculate_par_score


## 1. Verify DLS Table Curves
We calculate resource percentages using our DLS implementation and plot them to reconstruct the classic resource depletion curves.


In [ ]:
overs_grid = np.linspace(0, 50, 101)
wickets_list = list(range(10))

plt.figure(figsize=(10, 6))
for w in wickets_list:
    res_line = [get_resource_pct(u, w) for u in range(51)]
    plt.plot(list(range(51)), res_line, label=f'{w} wickets lost')

plt.title('DLS Standard Edition Resource Depletion Curves')
plt.xlabel('Overs Remaining')
plt.ylabel('Resource Percentage Remaining (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(FIGURES_DIR, 'dls_resource_depletion_curves.png'))
plt.show()


## 2. Validation Against Real DLS-Applied Matches
We load matches where DLS was applied, find their point of interruption (last ball bowled in innings 2 before match ended), compute the revised target using our baseline DLS lookup, and compare it to the target in Cricsheet metadata.


In [ ]:
df_meta = pd.read_parquet(os.path.join(INTERIM_DATA_DIR, 'match_meta.parquet'))
df_bbb = pd.read_parquet(os.path.join(INTERIM_DATA_DIR, 'ball_by_ball.parquet'))

dls_matches = df_meta[df_meta['has_dls_applied'] == True].copy()
print(f'Number of DLS applied matches in dataset: {len(dls_matches)}')

validation_records = []
for idx, row in dls_matches.iterrows():
    m_id = row['match_id']
    s1 = row['final_score_inn1']
    target_actual = row['dls_revised_target']
    
    if pd.isna(target_actual) or target_actual is None or target_actual <= 0:
        continue
        
    match_bbb = df_bbb[df_bbb['match_id'] == m_id]
    inn2_bbb = match_bbb[match_bbb['innings_no'] == 2]
    
    if inn2_bbb.empty:
        continue
        
    last_ball = inn2_bbb.iloc[-1]
    balls_bowled = last_ball['legal_balls_bowled']
    wickets_lost = last_ball['cumulative_wickets']
    
    inn1_bbb = match_bbb[match_bbb['innings_no'] == 1]
    inn1_balls = inn1_bbb.iloc[-1]['legal_balls_bowled'] if not inn1_bbb.empty else 120
    
    r1 = get_resource_pct(inn1_balls / 6.0, 0)
    
    overs_remaining = max(0.0, 20.0 - (balls_bowled / 6.0))
    r2_lost = get_resource_pct(overs_remaining, wickets_lost)
    r2_start = get_resource_pct(20, 0)
    r2 = r2_start - r2_lost
    
    computed_target = calculate_par_score(s1, r1, r2)
    
    validation_records.append({
        'match_id': m_id,
        'final_score_inn1': s1,
        'balls_bowled_inn2': balls_bowled,
        'wickets_lost_inn2': wickets_lost,
        'actual_target': target_actual,
        'computed_target': computed_target,
        'discrepancy': computed_target - target_actual
    })

df_val = pd.DataFrame(validation_records)
if not df_val.empty:
    df_val.to_csv(os.path.join(TABLES_DIR, 'dls_validation_discrepancy.csv'), index=False)
    mae = df_val['discrepancy'].abs().mean()
    print(f'Mean Absolute Discrepancy against recorded targets: {mae:.2f} runs')
    print(df_val.head(10))
else:
    print('No matches with target available for validation.')


## 3. Discrepancy Plot


In [ ]:
if not df_val.empty:
    plt.figure(figsize=(7, 7))
    plt.scatter(df_val['actual_target'], df_val['computed_target'], color='purple', alpha=0.7)
    max_val = max(df_val['actual_target'].max(), df_val['computed_target'].max())
    min_val = min(df_val['actual_target'].min(), df_val['computed_target'].min())
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='y = x')
    plt.title('DLS Target Validation: Computed vs. Actual Target')
    plt.xlabel('Actual Target (from Cricsheet)')
    plt.ylabel('Computed Target (our Standard Edition)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(FIGURES_DIR, 'dls_validation_scatter.png'))
    plt.show()
else:
    print('Validation plot skipped: no records.')


## Summary & Conclusion
We have coded the DLS Standard Edition resource lookup and validated it.
